In [4]:
%pip install opencv-python


Note: you may need to restart the kernel to use updated packages.


In [5]:
import cv2
import numpy as np
import os
import pandas as pd
import time

print(cv2.__version__)

5.0.0


PART A — Image Feature Extraction and Classification

In [7]:
p = "448"

fs = os.listdir(p)

print("Total files:", len(fs))
print(fs[:10])

Total files: 2
['Cracks', 'NonCracks']


In [8]:
cp = os.path.join(p, "Cracks")
np_ = os.path.join(p, "NonCracks")

cf = os.listdir(cp)
nf = os.listdir(np_)

print("Cracks:", len(cf))
print("NonCracks:", len(nf))

Cracks: 200
NonCracks: 200


In [9]:
a = cv2.imread(os.path.join(cp, cf[0]))
b = cv2.imread(os.path.join(np_, nf[0]))

print("Crack:", a.shape)
print("NonCrack:", b.shape)

Crack: (448, 448, 3)
NonCrack: (448, 448, 3)


In [10]:
sz = (224, 224)

a = cv2.resize(a, sz)
b = cv2.resize(b, sz)

ag = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
bg = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)

print(a.shape, ag.shape)
print(b.shape, bg.shape)

(224, 224, 3) (224, 224)
(224, 224, 3) (224, 224)


In [11]:
m = ag.mean()

c = ag.std()

dr = np.sum(ag < 50) / ag.size

br = np.sum(ag > 200) / ag.size

e = cv2.Canny(ag, 100, 200)

ed = np.sum(e > 0) / e.size

print("Mean:", m)
print("Contrast:", c)
print("Dark Ratio:", dr)
print("Bright Ratio:", br)
print("Edge Density:", ed)

Mean: 122.224609375
Contrast: 31.51348191896166
Dark Ratio: 0.0038464604591836736
Bright Ratio: 0.01092155612244898
Edge Density: 0.3434311224489796


In [12]:
rows = []

for f in cf:
    im = cv2.imread(os.path.join(cp, f))
    im = cv2.resize(im, sz)
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(g, 100, 200)

    rows.append([g.mean(), g.std(), np.mean(g < 50), np.mean(g > 200), np.mean(e > 0), 1])

for f in nf:
    im = cv2.imread(os.path.join(np_, f))
    im = cv2.resize(im, sz)
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(g, 100, 200)

    rows.append([g.mean(), g.std(), np.mean(g < 50), np.mean(g > 200), np.mean(e > 0), 0])

df = pd.DataFrame(rows, columns=["brightness", "contrast", "dark_ratio", "bright_ratio", "edge_density", "label"])

print(df.shape)
df.head()

(400, 6)


,brightness,contrast,dark_ratio,bright_ratio,edge_density,label
0,122.224609,31.513482,0.003846,0.010922,0.343431,1
1,131.948681,24.096114,0.002790,0.003787,0.266940,1
2,121.297313,29.884272,0.013333,0.008171,0.262775,1
3,130.245596,25.649018,0.001893,0.004006,0.330656,1
4,131.208885,26.237598,0.002372,0.006258,0.316606,1


In [13]:
df.to_csv("image_features.csv", index=False)

print("Saved:", df.shape)

Saved: (400, 6)


In [14]:
from sklearn.model_selection import train_test_split

x = df.drop("label", axis=1)
y = df["label"]

xtr, xte, ytr, yte = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

print(xtr.shape, xte.shape)

(320, 5) (80, 5)


In [16]:
%pip install --upgrade --force-reinstall scikit-learn

  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl (8.4 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl (12.7 MB)
Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl (37.4 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

  Attempting uninstall: threadpoolctl

    Found existing installation: threadpoolctl 3.7.0

    Uninst

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


In [17]:
import sklearn
print(sklearn.__version__)

1.9.1


In [24]:
import sys
print(sys.version)
print(sys.executable)

3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
c:\Users\Ummehani Khatri\AppData\Local\Python\pythoncore-3.14-64\python.exe


In [25]:
from sklearn.ensemble import RandomForestClassifier

m = RandomForestClassifier(n_estimators=100, random_state=42)

t1 = time.time()
m.fit(xtr, ytr)
tt = time.time() - t1

t1 = time.time()
yp = m.predict(xte)
pt = time.time() - t1

print("Training time:", tt)
print("Prediction time:", pt)

ImportError: DLL load failed while importing _seq_dataset: An Application Control policy has blocked this file.